### Домашнее задание 2 (часть 1): дообучение BERT

В этом задании вы получите эмбеддинги с помощью BERT, дообучите предобученную модель на новых данных и сравните возможности трансформеров с предыдущими архитектурами, учитывая их более высокие вычислительные затраты.


In [5]:
%pip install --quiet "transformers==4.57.6" "datasets==4.5.0" "accelerate==1.12.0" sentencepiece "protobuf>=6.33.5,<8" "scikit-learn>=1.5,<2"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.10.0 which is incompatible.
diff

In [7]:
import os
os.environ["WANDB_DISABLED"] = "true"

import torch
import transformers
import datasets
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, Trainer, TrainingArguments
print(transformers.__version__)


4.57.6


### Загрузка данных и модели


В этом задании используется **Quora Question Pairs (QQP)**.

Датасет содержит более 400 000 пар вопросов. Для каждой пары указано, являются ли вопросы перефразировками друг друга: речь идёт о дубликатах по смыслу, а не просто о связанных темах. Подробнее о данных можно прочитать [здесь](https://paperswithcode.com/dataset/quora-question-pairs).
Метки: 0 — не дубликаты, 1 — дубликаты.


In [8]:
qqp = datasets.load_dataset('SetFit/qqp')
print('\n')
print("Пример [0]:", qqp['train'][0])
print("Пример [3]:", qqp['train'][3])


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/70.8M [00:00<?, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl:   0%|          | 0.00/76.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Пример [0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Пример [3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [9]:
# Открытые проверки
for split in ['train', 'validation']:
    assert len(qqp[split]) > 0, f'{split}: датасет пуст.'
    assert set(qqp[split]['label']) == {0, 1}, f'{split}: ожидаются метки классов 0 и 1.'
    assert {'text1', 'text2', 'label'}.issubset(qqp[split].column_names), 'Сохраните оба вопроса и их метку.'


In [10]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)


tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

### Токенизация данных


Библиотека [datasets](https://huggingface.co/docs/datasets/en/index) позволяет применять функцию к данным с помощью `map`, как в функциональном программировании.

Что происходит с текстами в `qqp_preprocessed`?

- Исходные `text1` и `text2` преобразуются в числовые идентификаторы токенов с помощью соответствующего токенизатора.
- Для BERT тексты объединяются с разделителем `SEP`, а в начало добавляется токен `CLS`. Последовательность обрезается, если её длина превышает 128 токенов, или дополняется до 128 токенов, если она короче.
- Датасет `qqp_preprocessed` содержит:
    - _Идентификаторы токенов_ (`input_ids`): последовательность числовых идентификаторов.
    - _Маски внимания_ (`attention_mask`): бинарные маски, обозначающие позиции дополнения.
    - _Идентификаторы типов токенов_ (`token_type_ids`): различают токены из `text1` и `text2`.

__Обратите внимание!__ Маски внимания исключают позиции дополнения из внимания, но не отменяют все вычисления на этих позициях. `token_type_ids` могут отсутствовать.


In [11]:
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)


Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [12]:
# Открытые проверки
assert model.config.num_labels == 2, 'Ожидаются два логита классов.'
for split in ['train', 'validation']:
    assert len(qqp_preprocessed[split]) == len(qqp[split]), 'Токенизация должна сохранять число строк.'
    np.testing.assert_array_equal(qqp_preprocessed[split]['label'], qqp[split]['label'], err_msg='Сохраните метки и порядок строк.')
    sample = qqp_preprocessed[split][0]
    assert len(sample['input_ids']) == len(sample['attention_mask']) == MAX_LENGTH, 'Дополните или обрежьте последовательность до MAX_LENGTH.'
    assert set(sample['attention_mask']).issubset({0, 1}), 'Маска внимания должна содержать только 0 и 1.'
    assert 0 <= min(sample['input_ids']) <= max(sample['input_ids']) < model.config.vocab_size, 'Идентификаторы токенов должны принадлежать словарю этой модели.'


In [13]:
print(repr(qqp_preprocessed['train'][0]['input_ids'])[:100], "...")


[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Оценка качества (2 балла)

Мы выбрали готовую модель, обученную на QQP. Насколько хорошо она работает?

Один из способов это проверить — вычислить accuracy на валидационной выборке. Далее вы реализуете эту оценку.

Ниже приведён код, с которого можно начать:


Посмотрим на наши данные.


In [14]:
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)


In [15]:
for batch in val_loader:
     break  # Посмотрим на один батч.
print("Пример батча:", batch)

model.eval()
with torch.no_grad():
  predicted = model(
      input_ids=batch['input_ids'],
      attention_mask=batch['attention_mask'],
      token_type_ids=batch['token_type_ids']
  )

print('\nПредсказание (вероятности):', torch.softmax(predicted.logits, dim=1).data.numpy())


Пример батча: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

У модели одна классификационная голова с двумя выходными логитами — по одному

*   Новый пункт
*   Новый пункт

на каждый класс.


*__Ваша* задача__ — вычислить accuracy модели на валидационной выборке.
Наивная реализация может работать несколько часов. Используйте следующие оптимизации:

- запускайте модель на GPU с `no_grad`;
- используйте размер батча больше 1;
- начните с `num_workers=0`; при необходимости измерьте скорость с дополнительными процессами загрузки данных;
- при желании используйте [смешанную точность](https://pytorch.org/docs/stable/notes/amp_examples.html).


Хотя вычисления модели выполняются на GPU, загрузка данных с диска или из памяти и преобразование в нужный модели формат, например в тензоры, выполняются на CPU.

Недостаточная производительность CPU может замедлять весь процесс.


In [16]:
from tqdm import tqdm
import multiprocessing

cores = multiprocessing.cpu_count() # Число ядер процессора.
cores


2

In [17]:
# Переносим модель на GPU, если он доступен.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(device)

# Создаём DataLoader для валидационной выборки.
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=16,  # Увеличенный батч ускоряет обработку.
    shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers=0  # Подходит для Colab; дополнительные процессы загрузки данных — необязательная оптимизация.
)


cuda


In [19]:
# Оценка accuracy на валидации.
# TODO 1: замените каждое ... кодом для обработки всех батчей и вычисления accuracy.
model.eval()
correct_predictions = 0
total_predictions = 0
use_amp = device.type == 'cuda'


# labels - таргеты (batch_len,)
# idx - индекс строки в датасете
# input_ids - значения токенов
# token_type_ids - 0: первая последовательность токенов, padding; 1: вторая последовательность токенов
# attention_mask - маска, показывает какие токены считать

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Оценка качества"):
        # Перенесите входы модели и метки на устройство; исключите служебные поля строк.
        inputs = {
            'input_ids': batch['input_ids'].to(device),
            'attention_mask': batch['attention_mask'].to(device),
            'token_type_ids': batch['token_type_ids'].to(device)
        }
        labels = batch['labels'].to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            outputs = model(**inputs)

        # Получите предсказания классов и подсчитайте примеры, включая последний батч.

        predictions = torch.argmax(outputs.logits, dim=1)
        correct_predictions += (predictions == labels).sum().item()

        total_predictions += len(labels)


accuracy = correct_predictions / total_predictions  # Доля правильных ответов от 0 до 1.
print(f"\nAccuracy на валидации: {accuracy:.4f}")


Оценка качества: 100%|██████████| 2527/2527 [01:14<00:00, 34.06it/s]


Accuracy на валидации: 0.9084


In [20]:
assert isinstance(accuracy, (float, int, np.floating)) and np.isfinite(accuracy)
assert 0 <= accuracy <= 1
assert total_predictions == len(val_set), 'Обработайте каждый пример валидации ровно один раз.'
assert np.isclose(accuracy, correct_predictions / total_predictions)
baseline_accuracy = float(accuracy)
print('Опубликованная accuracy базовой модели на полной выборке — около 0.9084; другая длина или настройки могут изменить результат.')


Опубликованная accuracy базовой модели на полной выборке — около 0.9084; другая длина или настройки могут изменить результат.


### Дообучение модели (5 баллов)

В этой части вам нужно самостоятельно дообучить модель. Выберите предобученную модель __с архитектурой, отличной от оригинального BERT__, которую ранее не дообучали на QQP, в том числе в рамках GLUE или обучения на нескольких задачах. Рекомендуем [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Можно выбрать другую модель, опираясь на результаты публичных бенчмарков, например [GLUE](https://gluebenchmark.com/).

Вы можете написать цикл обучения самостоятельно или использовать `transformers.Trainer` (см. [пример](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Стремитесь к accuracy 92% на полной валидационной выборке. Оценка за вторую часть равна `C * clip((accuracy - 0.90) / (0.92 - 0.90), 0, 1)`, где `C` — баллы за корректность кода, от 0 до 5. При accuracy 90% и ниже оценка равна 0; при 92% и выше вы получаете все заработанные за код баллы. Сравните результат с готовой моделью BERT из первой части. Оценивайте обе модели в одинаковых условиях, обучайтесь только на тренировочной выборке и сохраните вывод сравнения. Если вы остановили обучение раньше, оцените текущую модель.


In [ ]:
# Освобождаем память GPU, занятую базовой моделью, перед загрузкой новой.
model.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Загрузите выбранную модель и её токенизатор, например DeBERTa-v3.
model_name = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = 2,
    dtype = torch.float32
)  # Загрузите с num_labels=2 и dtype=torch.float32 для обучения с AMP.

# Повторно токенизируйте qqp новым токенизатором; можно использовать функцию предобработки выше.

# Определите функцию токенизации пар или используйте уже написанную.
def preprocess_function(examples):
  result = tokenizer(
      examples['text1'], examples['text2'],
      padding = 'max_length', max_length = MAX_LENGTH, truncation = True
  )
  result['label'] = examples['label']
  return result

# Токенизируйте qqp, сохранив метки и порядок строк в выборках.
qqp_preprocessed = qqp.map(preprocess_function, batched=True)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [22]:
# Открытые проверки
assert model.config.num_labels == 2, 'Используйте два логита для классификации на два класса.'
assert model.config.model_type != 'bert', 'Выберите архитектуру, отличную от оригинального BERT.'
for split in ['train', 'validation']:
    raw, encoded = qqp[split], qqp_preprocessed[split]
    assert len(encoded) == len(raw), 'Сохраните исходный размер выборки.'
    np.testing.assert_array_equal(encoded['label'], raw['label'], err_msg='Сохраните метки и порядок строк.')
    probe_ids = [0, len(raw) // 2, len(raw) - 1]
    raw_probe = raw.select(probe_ids)
    expected = tokenizer(list(raw_probe['text1']), list(raw_probe['text2']), padding='max_length',
                         truncation=True, max_length=MAX_LENGTH)
    np.testing.assert_array_equal(encoded.select(probe_ids)['input_ids'], expected['input_ids'],
                                  err_msg='Повторно токенизируйте оба исходных вопроса новым токенизатором.')
accuracy = None  # После прерванного обучения не используйте прежнюю accuracy модели BERT.


In [ ]:
# Закончились лимиты, пришлось переносить чекпоинт на другой акк

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Подготавливаем тренировочную и валидационную выборки.
train_set = qqp_preprocessed['train']
val_set = qqp_preprocessed['validation']


# Задаём метрику оценки качества. При желании напишите собственную реализацию.
from sklearn.metrics import accuracy_score

# При использовании transformers.Trainer можно воспользоваться функцией ниже.
def compute_metrics(eval_pred):
    """
    Вычисляет метрики при обучении или оценке модели.
    Аргументы:
        eval_pred (tuple): Кортеж, содержащий:
            - logits (ndarray или torch.Tensor): Логиты модели для каждого примера
              в батче оценки. Форма: (batch_size, num_classes).
            - labels (ndarray или torch.Tensor): Истинные метки каждого примера.
              Форма: (batch_size,).
    Возвращает:
        dict: Словарь вычисленных метрик:
            - "accuracy" (float): Доля правильных ответов среди всех примеров.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, predictions) # Вычислите по eval_pred, не используя глобальную переменную accuracy.
    return {"accuracy": accuracy}

# Открытые проверки
metric_logits = np.array([[4, 1], [1, 4], [4, 1], [1, 4], [1, 4]])
metric_labels = np.array([0, 1, 0, 1, 0])
assert np.isclose(compute_metrics((metric_logits, metric_labels))['accuracy'], 0.8), 'Вычислите accuracy по примерам, выбрав предсказания вдоль измерения классов.'

# При желании напишите цикл обучения самостоятельно вместо transformers.Trainer.
# Начальное значение скорости обучения — 2e-5.
# При необходимости меняйте скорость обучения на порядок, например до 2e-4 или 2e-3.
# Трёх эпох обычно достаточно для дообучения без сильного забывания ранее изученного.
# Используйте регуляризацию weight_decay. Начните с 1e-2 и при необходимости измените значение.
# Можно выбирать лучшую модель по accuracy.

# Задайте параметры обучения без аргумента device: устройство выбирается автоматически.
# Оценивайте и сохраняйте модель каждые 1000 шагов оптимизатора; записывайте loss каждые 50 шагов.
training_args = TrainingArguments(
    output_dir = "/content/qqp_out",
    learning_rate = 2e-5,
    num_train_epochs = 3,
    weight_decay = 1e-2,
    warmup_steps = 2000,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 128,
    eval_strategy = "steps",
    eval_steps = 1000,
    save_strategy = "steps",
    save_steps = 1000,
    logging_steps = 50,
    save_total_limit = 2,
    load_best_model_at_end = True,
    metric_for_best_model = "accuracy",
    greater_is_better = True,
    fp16 = True,
    report_to = "none"
    )


# Оставляем веса в FP32 для AMP и очищаем градиенты от предыдущего неудачного запуска.
model.float()
model.zero_grad(set_to_none=True)

# Классификатор возвращает средний loss батча; Trainer учитывает накопление градиентов.
model.accepts_loss_kwargs = False

# Создаём Trainer.
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_set,
    eval_dataset = val_set,
    compute_metrics = compute_metrics
)

# Дообучаем модель.
trainer.train(resume_from_checkpoint="/content/qqp_out/checkpoint-15000")

# Оцените текущую модель на валидации и запишите результат в accuracy.
accuracy = trainer.evaluate()['eval_accuracy']
print(f"Accuracy на валидации: {accuracy:.4f}")

# загрузка на гугл диск финальной версии
import os
FINAL_DIR = "/content/drive/MyDrive/qqp_final"
try:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    trainer.save_model(FINAL_DIR)          # веса + config.json
    tokenizer.save_pretrained(FINAL_DIR)   # токенизатор
    print("Модель сохранена на Диск:", FINAL_DIR)
    print(os.listdir(FINAL_DIR))
except Exception as e:
    # запасной вариант: локальная копия, чтобы результат не потерялся
    trainer.save_model("/content/qqp_final")
    print("Не удалось записать на Диск:", e)
    print("Модель лежит локально: /content/qqp_final")


Step,Training Loss,Validation Loss,Accuracy
16000,0.165900,0.235064,0.912293
17000,0.153300,0.244258,0.913925
18000,0.157000,0.235163,0.914074
19000,0.154000,0.229839,0.915706
20000,0.145800,0.243363,0.914840
21000,0.162700,0.237077,0.915756
22000,0.158900,0.230497,0.915879
23000,0.117400,0.248538,0.918303


Step,Training Loss,Validation Loss,Accuracy
16000,0.165900,0.235064,0.912293
17000,0.153300,0.244258,0.913925
18000,0.157000,0.235163,0.914074
19000,0.154000,0.229839,0.915706
20000,0.145800,0.243363,0.914840
21000,0.162700,0.237077,0.915756
22000,0.158900,0.230497,0.915879
23000,0.117400,0.248538,0.918303
24000,0.122300,0.247705,0.917240
25000,0.147200,0.238688,0.918575


Accuracy на валидации: 0.9211
Модель сохранена на Диск: /content/drive/MyDrive/qqp_final
['config.json', 'model.safetensors', 'training_args.bin', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'spm.model', 'tokenizer.json']


In [24]:
assert isinstance(accuracy, (float, int, np.floating)) and np.isfinite(accuracy), 'Оцените дообученную модель; не используйте accuracy базовой модели.'
assert 0 <= accuracy <= 1
if 'trainer' in globals():
    assert trainer.state.global_step > 0, 'Запустите обучение перед итоговой оценкой.'
    logged_losses = [record[key] for record in trainer.state.log_history for key in ('loss', 'train_loss') if key in record]
    assert all(np.isfinite(value) for value in logged_losses), 'Значения функции потерь при обучении должны быть конечными.'
import json
comparison = {
    'baseline_accuracy': float(baseline_accuracy),
    'finetuned_accuracy': float(accuracy),
    'delta': float(accuracy - baseline_accuracy),
    'validation_examples': len(qqp['validation']),
}
print(f'Базовая модель BERT: {baseline_accuracy:.6f}; ваша модель: {accuracy:.6f}; прирост: {accuracy - baseline_accuracy:+.6f}')
print('Результат базовой модели BERT превышен:', accuracy > baseline_accuracy)

accuracy_factor = float(np.clip((accuracy - 0.90) / (0.92 - 0.90), 0, 1))
print(f'Коэффициент качества второй части: {accuracy_factor:.4f}; максимум баллов за вторую часть при этой accuracy: {5 * accuracy_factor:.4f}/5')
print('QQP_COMPARISON ' + json.dumps(comparison))


Базовая модель BERT: 0.908410; ваша модель: 0.921123; прирост: +0.012713
Результат базовой модели BERT превышен: True
Коэффициент качества второй части: 1.0000; максимум баллов за вторую часть при этой accuracy: 5.0000/5
QQP_COMPARISON {"baseline_accuracy": 0.9084095968340341, "finetuned_accuracy": 0.9211229285184269, "delta": 0.012713331684392815, "validation_examples": 40430}


Используйте валидационную выборку для оценки качества и выбора модели. Метки тестовой выборки QQP скрыты (`-1`), поэтому вычислить accuracy на ней локально нельзя.


### Поиск с помощью эмбеддингов BERT (3 балла, обязательная часть)

Теперь попробуем использовать эмбеддинги на основе BERT для поиска информации.

Обычный BERT и похожие модели не обучаются напрямую с контрастивной функцией потерь или функцией потерь для троек примеров, которые сближают эмбеддинги похожих текстов. Поэтому для поисковой системы лучше выбрать специализированную [модель для оценки сходства предложений](https://huggingface.co/models?pipeline_tag=sentence-similarity). Выберите модель, подходящую для этой задачи.

Как и в первом домашнем задании, вам нужно построить поисковую систему:

1. _Подготовьте базу эмбеддингов._ Из пар вопросов QQP возьмите только поле `text1` валидационной выборки (`validation`). Получите эмбеддинги выбранной моделью и сохраните их в `numpy.ndarray` для последующего использования. При желании можно использовать индекс [Faiss](https://github.com/facebookresearch/faiss).
2. _Реализуйте поиск вопросов, похожих на запрос._ Напишите функцию или класс для работы с базой. __Результат этой части оценивается по способности понятно вывести пять наиболее похожих вопросов Quora для произвольного нового запроса.__

Сравните, насколько лучше такой поиск учитывает смысл текстов по сравнению с предыдущим подходом.

Помимо вывода, возвращайте `(row_ids, cosine_scores)` по убыванию косинусной близости. Число результатов равно `min(top_k, len(database))`. Индексы строк базы должны соответствовать порядку исходных `text1`.


In [28]:
del trainer
model.cpu()
torch.cuda.empty_cache()

In [32]:
# TODO 4: выберите энкодер, создайте database и реализуйте поиск по запросу.
# Инициализируем модель и её токенизатор.
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

all_texts = list(qqp['validation']['text1'])

def get_embeddings(texts, model, batch_size = 128):
  embeddings = []

  with torch.no_grad():
    for i in tqdm(range(0, len(texts), batch_size)):
      batch_texts = texts[i:i + batch_size]
      tokens = tokenizer(
          batch_texts,
          padding = True, max_length = MAX_LENGTH, truncation = True,
          return_tensors = 'pt'
      )
      inputs = {
          'input_ids': tokens['input_ids'].to(device),
          'attention_mask': tokens['attention_mask'].to(device)
      }
      outputs = model(**inputs)

      mask = inputs['attention_mask'].unsqueeze(-1).float()
      emb = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)

      embeddings.append(emb.cpu().numpy())
  embeddings = np.concatenate(embeddings)
  embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

  return embeddings

# Получите по одной строке эмбеддингов NumPy для каждого исходного text1 в том же порядке.
# Используйте этот токенизатор и учитывайте дополнение при усреднении эмбеддингов токенов.
database = get_embeddings(all_texts, model)

def find_similar_questions(query, database, model, top_k = 5):
    """
    Находит и выводит top_k вопросов, наиболее похожих на запрос.

    Кодирует запрос, сравнивает его с заранее вычисленной базой
    эмбеддингов по косинусной близости и выводит наиболее
    похожие по смыслу вопросы.

    Аргументы:
        query (str): Поисковый запрос пользователя.
        database (np.ndarray): Двумерный массив эмбеддингов вопросов в базе.
        model: Модель, используемая со своим токенизатором для кодирования запроса.
        top_k (int): Число результатов для вывода.

    Возвращает:
        tuple: (row_ids, cosine_scores) по убыванию близости, помимо вывода.
    """
    query_emb = get_embeddings([query], model)[0]
    scores = database @ query_emb

    row_ids = np.argsort(-scores)[:top_k]
    cosine_scores = scores[row_ids]

    print(f"\nЗапрос: {query}")
    for i in range(len(row_ids)):
        print(f"{i + 1}. [{cosine_scores[i]:.4f}] {all_texts[row_ids[i]]}")

    return row_ids, cosine_scores

100%|██████████| 316/316 [00:16<00:00, 19.34it/s]


In [33]:
# Открытые проверки
assert isinstance(database, np.ndarray) and database.ndim == 2
assert database.shape[0] == len(all_texts) == len(qqp['validation'])
assert np.isfinite(database).all() and (np.linalg.norm(database, axis=1) > 0).all()
row_ids, cosine_scores = find_similar_questions("Are apples useful?", database, model, top_k=5)
row_ids, cosine_scores = np.asarray(row_ids), np.asarray(cosine_scores)
assert row_ids.ndim == cosine_scores.ndim == 1
assert len(row_ids) == len(cosine_scores) == min(5, len(database))
assert np.issubdtype(row_ids.dtype, np.integer) and len(set(row_ids.tolist())) == len(row_ids)
assert ((row_ids >= 0) & (row_ids < len(database))).all()
assert np.isfinite(cosine_scores).all() and (np.abs(cosine_scores) <= 1 + 2e-3).all()
assert (np.diff(cosine_scores) <= 1e-6).all(), 'Возвращайте результаты по убыванию косинусной близости.'


100%|██████████| 1/1 [00:00<00:00, 144.62it/s]


Запрос: Are apples useful?
1. [0.5971] Is it healthy to eat an apple at night?
2. [0.5638] Can hamsters eat apples?
3. [0.5147] What is the first thing that comes to your mind when you hear the word apple?
4. [0.5147] What is the first thing that comes to your mind when you hear the word apple?
5. [0.5142] What makes Apple the most valuable brand?


In [34]:
queries = [
    "How can I lose weight fast?",
    "What is the best way to learn programming?",
    "Why is the sky blue?",

]
for q in queries:
    find_similar_questions(q, database, model, top_k = 5)

100%|██████████| 1/1 [00:00<00:00, 77.42it/s]



Запрос: How can I lose weight fast?
1. [0.9920] How can I lose my weight fast?
2. [0.9264] What are the best ways to lose weight fast?
3. [0.9264] What are the best ways to lose weight fast?
4. [0.9264] What are the best ways to lose weight fast?
5. [0.9264] What are the best ways to lose weight fast?


100%|██████████| 1/1 [00:00<00:00, 57.71it/s]



Запрос: What is the best way to learn programming?
1. [0.9987] What is the the best way to learn programming?
2. [0.9253] What is best way to start learning programming?
3. [0.9253] What is best way to start learning programming?
4. [0.9238] What is the best way to learn a programming language?
5. [0.8665] What is the best way to learn and master a computer programming language?


100%|██████████| 1/1 [00:00<00:00, 51.33it/s]


Запрос: Why is the sky blue?
1. [0.9657] Why is that the sky is so blue?
2. [0.9071] Why does the Earth have blue sky?
3. [0.6732] What color is the sky on Mars?
4. [0.6186] Why does, sometimes, the sky gets orange when it rains at night?
5. [0.5711] Why can I see both the moon and the sun in the sky?
